# Kaggle Pipeline: TD3 + DDPG (Batched)

This notebook runs both algorithms sequentially for a small comparison batch.
Use `BATCH_INDEX = 0..1` to control which comparison slice runs.
It is resume-safe and GPU-compatible.

## 1. Kaggle Setup + Repository

Sets up `/kaggle/working`, clones the repo, and ensures logs/models/results are stored under `/kaggle/working`.

In [ ]:
BASE_DIR = "/kaggle/working"

!git clone https://github.com/adityagangwani30/TD3-Car-Game.git
%cd TD3-Car-Game

import os
import shutil
from pathlib import Path

LOGS_DIR = Path(BASE_DIR) / "logs"
MODELS_DIR = Path(BASE_DIR) / "models"
RESULTS_DIR = Path(BASE_DIR) / "results"

for d in [LOGS_DIR, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Route repository outputs to Kaggle working directories.
repo_root = Path.cwd()
for name, target in [("logs", LOGS_DIR), ("models", MODELS_DIR), ("results", RESULTS_DIR)]:
    link_path = repo_root / name
    if link_path.exists() or link_path.is_symlink():
        if link_path.is_symlink():
            link_path.unlink()
        elif link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()
    os.symlink(target, link_path)

print("BASE_DIR:", BASE_DIR)
print("Working directory:", os.getcwd())
print("logs ->", (repo_root / "logs").resolve())
print("models ->", (repo_root / "models").resolve())
print("results ->", (repo_root / "results").resolve())

## 2. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 3. GPU Setup

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("device =", device)

In [ ]:
MAX_EPISODES = 2000
MAX_STEPS_PER_EPISODE = 300
SEEDS = [0, 42, 123]
COMPARE_BATCHES = [
    (0, 1),   # experiment 0 only
    (3, 1),   # experiment 3 only
]
BATCH_INDEX = 0  # user can change (0–1)
ALGOS = ["td3", "ddpg"]
EXPERIMENT_TAGS = [f"R{r}_N{n}" for r in range(1, 5) for n in range(1, 4)]
start_idx, num_exp = COMPARE_BATCHES[BATCH_INDEX]
batch_experiments = EXPERIMENT_TAGS[start_idx:start_idx + num_exp]
print(f"Notebook role: BOTH | Batch {BATCH_INDEX} -> start={start_idx}, count={num_exp}")

## 4. Run TD3 Experiments (12 Runs, 2000 Episodes Each)

In [ ]:
for algo in ALGOS:
    for experiment in batch_experiments:
        for seed in SEEDS:
            print(f"[{algo.upper()}][{experiment}][Seed {seed}] Running...")
            !python run_experiments.py \
                --algo {algo} \
                --max-episodes 2000 \
                --max-steps 300 \
                --start-index {start_idx} \
                --max-experiments {num_exp} \
                --seed {seed} \
                --headless \
                --resume

## 5. Run DDPG Experiments (12 Runs, 2000 Episodes Each)

## 6. Plot Results

In [ ]:
!python plot_metrics.py --compare-algos

## 7. Save Final Results

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/final_results', 'zip', '/kaggle/working/results')
print("Saved: /kaggle/working/final_results.zip")